In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from timm import create_model, list_models
import torch.optim as optim
from tqdm import tqdm
import os


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/envs/3d-recon-ai/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/envs/3d-recon-ai/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/envs/3d-recon-ai/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.

In [4]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [5]:
device

device(type='cpu')

In [15]:
model_list = list_models('convnextv2_*')
print(model_list)

['convnextv2_atto', 'convnextv2_base', 'convnextv2_femto', 'convnextv2_huge', 'convnextv2_large', 'convnextv2_nano', 'convnextv2_pico', 'convnextv2_small', 'convnextv2_tiny']


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset

class BiPlanarXrayDataset(Dataset):
    def __init__(self, patient_dirs, transform=None):
        self.patient_dirs = patient_dirs
        self.transform = transform

    def __len__(self):
        return len(self.patient_dirs)

    def __getitem__(self, idx):
        patient_path = self.patient_dirs[idx]
        front = Image.open(os.path.join(patient_path, 'front.png')).convert('L')
        side = Image.open(os.path.join(patient_path, 'side.png')).convert('L')

        if self.transform:
            front = self.transform(front)
            side = self.transform(side)

        return front, side



In [ ]:

# List all patient folders
root_dir = 'data'
all_patients = [os.path.join(root_dir, d) for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]

# Split into train and val
train_dirs, val_dirs = train_test_split(all_patients, test_size=0.2, random_state=42)

# Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # Convert 1-channel X-ray to 3
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])


# Create dataset instances
train_dataset = BiPlanarXrayDataset(train_dirs, transform=transform)
val_dataset = BiPlanarXrayDataset(val_dirs, transform=transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


In [ ]:
# Load Pretrained ConvNeXt
model = create_model('convnextv2_small', pretrained=True, num_classes=2)  # adjust num_classes
model = model.to(device)

# Optional: Freeze some layers (for partial fine-tuning)
for name, param in model.named_parameters():
    if "stages.3" in name or "head" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# Fine-Tuning Loop
for epoch in range(5):  # adjust epochs
    model.train()
    total_loss = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# ✅ Done fine-tuning
torch.save(model.state_dict(), 'convnext_finetuned_xray.pth')


In [ ]:
# Load fine-tuned model again (if needed)
model = create_model('convnext_small', pretrained=True)
model.load_state_dict(torch.load('convnext_finetuned_xray.pth'))
model.eval()
model = model.to(device)

# Extractor wrapper (gets 768-dim vector)
class FeatureExtractor(nn.Module):
    def __init__(self, convnext_model):
        super().__init__()
        self.features = convnext_model.forward_features
        self.norm = convnext_model.norm  # apply final normalization

    def forward(self, x):
        x = self.features(x)  # shape: [B, 768]
        x = self.norm(x)
        return x

extractor = FeatureExtractor(model).to(device)

# Run on one image (or a batch)
import PIL.Image as Image
from torchvision import transforms

image_path = ''
img = Image.open(image_path).convert('L')
img = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    feature_vector = extractor(img)  # shape: [1, 768]
    print(feature_vector.shape)
